# 🎬 《霧港館的第六個人》美術館第一人稱高畫質過場影片生成器（Colab A100 GPU）

本 Notebook 專為 **Google Colab A100 GPU** 設計，讀取 `assets/scene_*.jpg` 美術館插圖，使用 **Wan2.1 / Wan2.2 14B 旗艦級開源圖生影片模型**，以 **720P 高畫質（1280×720）** 保留原本 2D 懸疑插畫筆觸並生成流暢的第一人稱推進運鏡 `.mp4` 影片，自動存入 Google Drive `MyDrive/Final-Project-Videos/`。

### 模型選擇說明
- **`Wan2.1-I2V-14B-720P`（預設強烈推薦）**：14B 旗艦圖生影片模型，原生 **1280×720 HD**＋FP32 高精細 VAE，完整保留 2D 線條與光影細節，運鏡真實流暢（A100 40GB 約 2～3 分鐘／支）。
- **`Wan2.2-I2V-A14B`（27B MoE 電影旗艦）**：最新雙專家 MoE 架構，對風格化插畫與鏡頭穩定度最強（檔案較大，首次下載約需 5～8 分鐘）。
- **`Wan2.1-I2V-14B-480P`（快速預覽版）**：832×480，生成速度約快 2.5 倍，適合快速試跑 Prompt。

> 💡 建議先勾選第 4 格的 **`TEST_ONE_SCENE_FIRST = True`** 只跑 1 支 `scene_museum.mp4` 確認畫質滿意，再取消勾選一口氣跑完 6 支！

In [ ]:
#@title 1. 檢查 A100 GPU 環境並掛載 Google Drive
import os
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), "❌ 未偵測到 GPU！請到「執行階段 → 變更執行階段類型」選擇 A100 GPU。"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"✅ 已連線 GPU：{gpu_name}（VRAM: {vram_gb:.1f} GB）")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

drive.mount("/content/drive", force_remount=False)
OUTPUT_DIR = Path("/content/drive/MyDrive/Final-Project-Videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Google Drive 輸出資料夾已就緒：{OUTPUT_DIR}")

In [ ]:
#@title 2. 安裝 Diffusers、Accelerate 與相容版 SentencePiece（修正 Colab Protobuf 相容性）
!pip install -q --upgrade "diffusers>=0.32.2" "transformers>=4.49.0" "accelerate>=1.2.0" "sentencepiece==0.2.2" ftfy imageio imageio-ffmpeg einops av
!ffmpeg -version | head -n 2

import sys, importlib
for mod in [k for k in list(sys.modules) if k == "sentencepiece" or k.startswith("sentencepiece.")]:
    del sys.modules[mod]
importlib.invalidate_caches()

import diffusers, transformers, sentencepiece
from sentencepiece import sentencepiece_model_pb2
print(f"✅ diffusers {diffusers.__version__} | transformers {transformers.__version__} | sentencepiece {sentencepiece.__version__}")

In [ ]:
#@title 3. 複製 erickuo0707/Final-Project 並讀取 assets/scene_*.jpg
import subprocess
from pathlib import Path
from PIL import Image

REPO_URL = "https://github.com/erickuo0707/Final-Project.git"
REPO_DIR = Path("/content/Final-Project")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)

ASSETS_DIR = REPO_DIR / "assets"
scene_files = sorted(ASSETS_DIR.glob("scene_*.jpg"))
print(f"🖼️ 找到 {len(scene_files)} 張場景圖：")
for p in scene_files:
    with Image.open(p) as im:
        print(f"  - {p.name:28s} ({im.width}x{im.height})")

In [ ]:
#@title 4. 高畫質模型與動態運鏡設定（預設 Wan2.1-I2V-14B-720P）
MODEL_CHOICE = "Wan2.1-I2V-14B-720P" #@param ["Wan2.1-I2V-14B-720P", "Wan2.2-I2V-A14B", "Wan2.1-I2V-14B-480P"]
DURATION_SECONDS = 2.5 #@param {type:"slider", min:2.0, max:3.0, step:0.5}
INFERENCE_STEPS = 35 #@param {type:"slider", min:25, max:50, step:5}
TEST_ONE_SCENE_FIRST = False #@param {type:"boolean"}
INCLUDE_COURTROOM = False #@param {type:"boolean"}
OVERWRITE_EXISTING = True #@param {type:"boolean"}
SEED = 1842 #@param {type:"integer"}

NEGATIVE_PROMPT = (
    "static image, frozen camera, no motion, blurry, low quality, muddy textures, smeared lines, "
    "photorealistic shift, deformed perspective, flickering text, warped walls, bright daylight, "
    "extra characters, watermark, compression artifacts"
)

STYLE_ANCHOR = (
    "Maintain exact dark noir 2D visual novel illustration art style with crisp clean dark ink outlines "
    "and rich painterly shading throughout every frame. "
)

# 強調明確的 3D 鏡頭推進、視差移動與戲劇光影動態，避免畫面靜止或糊掉
SCENE_SPECS = [
    {
        "stem": "scene_museum",
        "label": "2F B 展廳（《海霧港灣》主牆）",
        "prompt": (
            STYLE_ANCHOR
            + "Dynamic first-person POV walking steadily forward down the dark marble gallery toward the large golden-framed seascape painting. "
            "Clear forward dolly-in camera motion with realistic walking parallax across the red velvet rope and polished reflective floor. "
            "Warm overhead golden spotlight beams shimmer while luminous golden dust motes drift across the dark hall."
        ),
    },
    {
        "stem": "scene_security",
        "label": "1F 保全中控區",
        "prompt": (
            STYLE_ANCHOR
            + "First-person POV walking forward into the nocturnal museum security control room with smooth forward dolly camera movement and subtle walking head-bob. "
            "The dual CCTV monitors on the left desk cast animated flickering cyan light, red and amber LED indicators blink rhythmically across the metal alarm panel on the right wall, "
            "and dust specks float beneath the warm center spotlight."
        ),
    },
    {
        "stem": "scene_basement",
        "label": "B1 地下器材室",
        "prompt": (
            STYLE_ANCHOR
            + "First-person POV walking steadily forward into the dim underground janitor storage room toward the wooden work desk on the right. "
            "Smooth camera push-in creating strong depth parallax past the cleaning cart on the left. "
            "The warm articulated desk lamp glows brightly over the open shift logbook while fine dust particles drift through the cone of light."
        ),
    },
    {
        "stem": "scene_restoration",
        "label": "B1 修復室 R-07",
        "prompt": (
            STYLE_ANCHOR
            + "First-person POV walking forward toward the wooden easel in the center of the hidden museum restoration room with clear dolly-in camera parallax past the pigment table on the left. "
            "The suspended purple UV tube lamp pulses with eerie violet fluorescence across the painting surface while the archival film lightbox glows steadily on the right wall."
        ),
    },
    {
        "stem": "scene_loading_dock",
        "label": "1F 西側裝卸區",
        "prompt": (
            STYLE_ANCHOR
            + "First-person POV walking forward into the spacious nocturnal museum loading dock toward the clipboard desk and wooden climate-controlled art crate on the rolling cart. "
            "Noticeable forward camera glide with perspective shift across the tire tracks on the concrete floor, overhead industrial pendant lamp gently swaying and casting shifting warm shadows."
        ),
    },
    {
        "stem": "scene_curator_office",
        "label": "2F 策展人辦公室",
        "prompt": (
            STYLE_ANCHOR
            + "First-person POV stepping forward into the quiet mahogany-paneled curator office, smooth forward dolly-in toward the green banker lamp, open auction catalog, and ornate brass wooden lockbox on the desk. "
            "Outside the tall arched window on the right, cold harbor night fog visibly drifts past distant ship silhouettes while warm emerald light shimmers on the wood desk."
        ),
    },
]

if INCLUDE_COURTROOM:
    SCENE_SPECS.append(
        {
            "stem": "scene_courtroom",
            "label": "市立霧港地方法院（選用）",
            "prompt": (
                STYLE_ANCHOR
                + "First-person POV walking forward down the center aisle of a grand wooden courtroom toward the judge bench, "
                "strong forward dolly camera parallax and glowing dust motes drifting through dramatic ceiling spotlights."
            ),
        }
    )

if TEST_ONE_SCENE_FIRST:
    SCENE_SPECS = SCENE_SPECS[:1]

print(f"🎬 已設定模型：{MODEL_CHOICE}｜目標長度：{DURATION_SECONDS}s｜步數：{INFERENCE_STEPS}｜待生成：{len(SCENE_SPECS)} 支")

In [ ]:
#@title 5. 載入高畫質 Wan2.1 / Wan2.2 14B 圖生影片管線
import gc
import importlib
import subprocess
import sys
import torch

# 自動確保 sentencepiece 與 Colab protobuf 6.x 完全相容
try:
    from sentencepiece import sentencepiece_model_pb2 as _sp_pb2
except Exception as _sp_err:
    print(f"🔧 自動修補 SentencePiece 相容性（{_sp_err}）...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "sentencepiece==0.2.2"], check=True)
    for _m in [k for k in list(sys.modules) if k == "sentencepiece" or k.startswith("sentencepiece.")]:
        del sys.modules[_m]
    importlib.invalidate_caches()
    from sentencepiece import sentencepiece_model_pb2 as _sp_pb2
print("✅ SentencePiece tokenizer 相容性檢查通過")

pipe = None
gc.collect()
torch.cuda.empty_cache()

from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from transformers import CLIPVisionModel

def _load_pretrained(cls, model_id, dtype=torch.bfloat16, **kwargs):
    try:
        return cls.from_pretrained(model_id, dtype=dtype, **kwargs)
    except TypeError:
        return cls.from_pretrained(model_id, torch_dtype=dtype, **kwargs)

VIDEO_FPS = 16
k = max(8, round((DURATION_SECONDS * VIDEO_FPS - 1) / 4))
NUM_FRAMES = 4 * k + 1  # Wan 系列規定 4*k + 1（例如 41 幀 @ 16fps = 2.56s）

if MODEL_CHOICE == "Wan2.1-I2V-14B-720P":
    MODEL_ID = "Wan-AI/Wan2.1-I2V-14B-720P-Diffusers"
    TARGET_W, TARGET_H = 1280, 720
    FLOW_SHIFT = 5.0
    GUIDANCE_SCALE = 5.0
    print(f"⬇️ 正在載入 720P 旗艦模型 {MODEL_ID}（首次下載約 28GB，請稍候 2～4 分鐘）...")
    image_encoder = _load_pretrained(CLIPVisionModel, MODEL_ID, dtype=torch.float32, subfolder="image_encoder")
    vae = _load_pretrained(AutoencoderKLWan, MODEL_ID, dtype=torch.float32, subfolder="vae")
    pipe = _load_pretrained(
        WanImageToVideoPipeline,
        MODEL_ID,
        dtype=torch.bfloat16,
        vae=vae,
        image_encoder=image_encoder,
    )
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=FLOW_SHIFT)
    pipe.enable_model_cpu_offload()

elif MODEL_CHOICE == "Wan2.2-I2V-A14B":
    MODEL_ID = "Wan-AI/Wan2.2-I2V-A14B-Diffusers"
    TARGET_W, TARGET_H = 1280, 720
    GUIDANCE_SCALE = 3.5
    print(f"⬇️ 正在載入 27B MoE 電影級旗艦 {MODEL_ID}（首次下載較久，請稍候）...")
    vae = _load_pretrained(AutoencoderKLWan, MODEL_ID, dtype=torch.float32, subfolder="vae")
    pipe = _load_pretrained(WanImageToVideoPipeline, MODEL_ID, dtype=torch.bfloat16, vae=vae)
    pipe.enable_model_cpu_offload()

elif MODEL_CHOICE == "Wan2.1-I2V-14B-480P":
    MODEL_ID = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"
    TARGET_W, TARGET_H = 832, 480
    FLOW_SHIFT = 3.0
    GUIDANCE_SCALE = 5.0
    print(f"⬇️ 正在載入快速版 {MODEL_ID} ...")
    image_encoder = _load_pretrained(CLIPVisionModel, MODEL_ID, dtype=torch.float32, subfolder="image_encoder")
    vae = _load_pretrained(AutoencoderKLWan, MODEL_ID, dtype=torch.float32, subfolder="vae")
    pipe = _load_pretrained(
        WanImageToVideoPipeline,
        MODEL_ID,
        dtype=torch.bfloat16,
        vae=vae,
        image_encoder=image_encoder,
    )
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=FLOW_SHIFT)
    pipe.enable_model_cpu_offload()
else:
    raise ValueError(f"Unsupported MODEL_CHOICE: {MODEL_CHOICE}")

if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
    pipe.vae.enable_tiling()

actual_sec = NUM_FRAMES / VIDEO_FPS
print(f"✅ 模型就緒！{MODEL_CHOICE}｜解析度：{TARGET_W}x{TARGET_H}｜幀數：{NUM_FRAMES} 幀 @ {VIDEO_FPS}fps（{actual_sec:.2f} 秒）")

In [ ]:
#@title 6. 批次生成 720P 高畫質過場影片並自動轉碼存至 Google Drive
import time
import subprocess
from pathlib import Path
from PIL import Image, ImageOps
from diffusers.utils import export_to_video

TEMP_DIR = Path("/content/tmp_videos")
TEMP_DIR.mkdir(parents=True, exist_ok=True)


def prepare_image(img_path: Path, width: int, height: int) -> Image.Image:
    im = Image.open(img_path).convert("RGB")
    return ImageOps.fit(im, (width, height), method=Image.Resampling.LANCZOS)


def transcode_web_mp4(raw_mp4: Path, out_mp4: Path):
    # 高畫質網頁相容編碼（CRF 17 + slow preset，保留暗部細節與 2D 清晰線條）
    cmd = [
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", str(raw_mp4),
        "-c:v", "libx264",
        "-preset", "slow",
        "-pix_fmt", "yuv420p",
        "-profile:v", "high",
        "-crf", "17",
        "-movflags", "+faststart",
        "-an",
        str(out_mp4),
    ]
    subprocess.run(cmd, check=True)


generated_paths = []
for idx, spec in enumerate(SCENE_SPECS, start=1):
    stem = spec["stem"]
    src_img = ASSETS_DIR / f"{stem}.jpg"
    final_mp4 = OUTPUT_DIR / f"{stem}.mp4"

    if not src_img.exists():
        print(f"⚠️ 找不到來源圖片：{src_img}，略過。")
        continue
    if final_mp4.exists() and not OVERWRITE_EXISTING:
        size_kb = final_mp4.stat().st_size / 1024
        print(f"⏭️ [{idx}/{len(SCENE_SPECS)}] 已存在於 Drive：{final_mp4.name}（{size_kb:.0f} KB），略過。")
        generated_paths.append(final_mp4)
        continue

    print(f"🎬 [{idx}/{len(SCENE_SPECS)}] 開始生成 {spec['label']}（{stem}.mp4｜{TARGET_W}x{TARGET_H}）...")
    t0 = time.time()
    init_image = prepare_image(src_img, TARGET_W, TARGET_H)
    generator = torch.Generator(device="cuda").manual_seed(SEED + idx)

    call_kwargs = dict(
        image=init_image,
        prompt=spec["prompt"],
        negative_prompt=NEGATIVE_PROMPT,
        width=TARGET_W,
        height=TARGET_H,
        num_frames=NUM_FRAMES,
        num_inference_steps=INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    )
    if MODEL_CHOICE == "Wan2.2-I2V-A14B":
        call_kwargs["guidance_scale_2"] = GUIDANCE_SCALE

    result = pipe(**call_kwargs)
    frames = result.frames[0]
    raw_mp4 = TEMP_DIR / f"{stem}_raw.mp4"
    export_to_video(frames, str(raw_mp4), fps=VIDEO_FPS)
    transcode_web_mp4(raw_mp4, final_mp4)

    elapsed = time.time() - t0
    size_kb = final_mp4.stat().st_size / 1024
    print(f"   ✅ 完成！耗時 {elapsed:.1f}s｜大小 {size_kb:.0f} KB｜已存至 {final_mp4}")
    generated_paths.append(final_mp4)
    torch.cuda.empty_cache()

print(f"\n🎉 全部處理完成！共 {len(generated_paths)} 支高畫質影片已保存在：{OUTPUT_DIR}")

In [ ]:
#@title 7. 在 Colab 內即時預覽 Google Drive 內的過場影片
import base64
from IPython.display import HTML, display

mp4_files = sorted(OUTPUT_DIR.glob("*.mp4"))
if not mp4_files:
    print("尚未找到任何 .mp4 影片，請先執行上一個生成儲存格。")
else:
    cards = []
    for mp4 in mp4_files:
        b64 = base64.b64encode(mp4.read_bytes()).decode("ascii")
        size_kb = mp4.stat().st_size / 1024
        cards.append(
            f'''<div style="width:460px;background:#141820;border:1px solid #8a6e40;border-radius:12px;padding:10px;color:#f3e3be;">
              <div style="font-weight:700;margin-bottom:6px;">🎬 {mp4.name} ({size_kb:.0f} KB)</div>
              <video controls autoplay loop muted playsinline style="width:100%;border-radius:8px;" src="data:video/mp4;base64,{b64}"></video>
            </div>'''
        )
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:14px;">{"".join(cards)}</div>'))

## 🚀 如何啟用到 Vercel 網頁版？

1. 開啟 Google Drive 的 **`MyDrive/Final-Project-Videos/`**，下載剛剛生成的 `scene_*.mp4` 影片。
2. 將這些 `.mp4` 檔案複製到 GitHub 專案的 **`assets/videos/`** 資料夾（例如 `assets/videos/scene_museum.mp4`、`assets/videos/scene_security.mp4`…）。
3. 提交並推送到 `main` 分支：
   ```bash
   git add assets/videos/*.mp4
   git commit -m "feat: add AI-generated 720p first-person museum transition videos"
   git push origin main
   ```
4. Vercel 自動重新部署後，玩家點擊門口／樓梯熱區或切換場景時，`web/app.js` 會**自動偵測並優先播放 `assets/videos/` 裡的 `.mp4` 過場影片**。